# Baseline & Spectral Pollution

Setup: W0 (Zero Wind) + G1 (Square) and G2 (L-Shape).


### Required Libraries

In [1]:
import numpy as np       #numpy
import scipy.sparse as sp #sparse matrices
import matplotlib.pyplot as plt #matplot
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogLocator, LogFormatterMathtext
from firedrake import *
from netgen.occ import *
from ngsolve.webgui import Draw
import math

firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


## Mesh Generation & Configuration

In [2]:
def make_Lshape_mesh(maxh=0.1):
    """
    Build the L-shaped domain and mesh it with Netgen.

    Parameters:
    maxh (float): maximum mesh size. No triangle edge will be longer than maxh.

    Returns:
    mesh: a Firedrake Mesh of the L-shaped domain.
    """
    # Axes((0,0,0), n=Z, h=X): Sets up a local coordinate system.
    #   - Origin is at (0,0,0).
    #   - Normal vector (n) is the Z-axis (meaning the plane is the XY-plane).
    #   - Horizontal axis (h) is the X-axis.
    # WorkPlane(...): Creates a 2D sketching plane using those axes.
    # Rectangle(1,2): Draws a rectangle from (0,0) to (1,2) on that plane.
    # Face(): Converts the hollow rectangle outline into a filled 2D surface.
    rect1 = WorkPlane(Axes((0, 0, 0), n=Z, h=X)).Rectangle(1, 2).Face()

    # Rectangle(2,1): Draws a rectangle from (0,0) to (2,1) on that plane.
    rect2 = WorkPlane(Axes((0, 0, 0), n=Z, h=X)).Rectangle(2, 1).Face()

    # join rect1 and rect2 together into a single continuous L-shaped 2D face,
    # automatically removing the overlapping internal edges.
    L = rect1 + rect2  # this is an OCC object

    # OCCGeometry(...): Wraps the OpenCascade shape (L) into a Netgen geometry object.
    # dim=2: Explicitly tells Netgen that this is a 2D surface geometry, not 3D.
    geo = OCCGeometry(L, dim=2)

    # GenerateMesh(...): Calls Netgen's meshing algorithm to divide the shape
    # into triangles.
    ngmsh = geo.GenerateMesh(maxh=maxh)

    # Mesh(...): Wraps the raw Netgen mesh into a Firedrake Mesh object.
    return Mesh(ngmsh)


def make_squareshape_mesh(maxh=0.1, side=1.0):
    """
    Build a square domain and mesh it with Netgen.

    The square has no reentrant corner, so it serves as the smooth-domain
    control against the L-shape.

    Parameters:
    maxh (float): maximum mesh size. No triangle edge will be longer than maxh.
    side (float): side length of the square.

    Returns:
    mesh: a Firedrake Mesh of the square domain.
    """
    # Same local coordinate system as the L-shape: origin at (0,0,0), normal
    # along Z so we sketch in the XY-plane, horizontal axis along X.
    # Rectangle(side, side): a square from (0,0) to (side,side).
    # Face(): Converts the hollow outline into a filled 2D surface.
    square = WorkPlane(Axes((0, 0, 0), n=Z, h=X)).Rectangle(side, side).Face()

    # OCCGeometry(...): Wraps the OpenCascade shape into a Netgen geometry object.
    # dim=2: Explicitly tells Netgen that this is a 2D surface geometry, not 3D.
    geo = OCCGeometry(square, dim=2)

    # GenerateMesh(...): Calls Netgen's meshing algorithm to divide the shape
    # into triangles.
    ngmsh = geo.GenerateMesh(maxh=maxh)

    # Mesh(...): Wraps the raw Netgen mesh into a Firedrake Mesh object.
    return Mesh(ngmsh)


### Test FE Spaces

In [5]:
mesh = make_squareshape_mesh(maxh=0.1, side=np.pi)
V_nodal = VectorFunctionSpace(mesh, "CG", 1)
Q_nodal = FunctionSpace(mesh, "CG", 1)
W_nodal = V_nodal * Q_nodal


V_A = FunctionSpace(mesh, "N1curl", 1)
Q_A = FunctionSpace(mesh, "CG", 1)
W_A = V_A * Q_A

### Weak Forms & Boundary Conditions

In [32]:
W = W_A


(v, p) = TrialFunctions(W)
(w, q) = TestFunctions(W)
a = (inner(curl(v), curl(w)) - inner(grad(p), w) - inner(v, grad(q))) * dx
m = inner(v, w) * dx


normal = FacetNormal(mesh)
tangent = as_vector((-normal[1], normal[0]))

penalty_parameter = 1e8
a += penalty_parameter * inner(dot(v, tangent), dot(w, tangent)) * ds
bcs = []
#bcs = DirichletBC(W.sub(0), Constant((0.0, 0.0)), "on_boundary")

### Eigensolver Configuration

In [33]:
opts = {
    "eps_target": 1.0,
    "st_type": "sinvert",
    "st_pc_type": "lu",
    "st_pc_factor_mat_solver_type": "mumps",
    "st_mat_mumps_icntl_24": 1,
    "st_mat_mumps_icntl_25": 0,
}

prob = LinearEigenproblem(A=a, M=m, bcs=bcs)
solver = LinearEigensolver(prob, n_evals=30, solver_parameters=opts)

nconv = solver.solve()
print(f"Converged: {nconv}")

for i in range(nconv):
    lam = solver.eigenvalue(i)
    print(f"{i}: {lam}")

Converged: 31
0: (5.305400062822016+0j)
1: (9.33940907737701+0j)
2: (9.340453811170136+0j)
3: (13.002960387904938+0j)
4: (15.626871012960974+0j)
5: (16.92808779948407+0j)
6: (19.228427917122055+0j)
7: (19.23266742929419+0j)
8: (24.966171599300466+0j)
9: (24.97674619864397+0j)
10: (24.997843742391712+0j)
11: (27.311575888565+0j)
12: (28.30379754765019+0j)
13: (33.148001861106124+0j)
14: (33.15162683813406+0j)
15: (35.41419072408082+0j)
16: (36.761329561031495+0j)
17: (38.56894877688534+0j)
18: (38.582097620008625+0j)
19: (40.98324687740005+0j)
20: (43.18187048298314+0j)
21: (44.11728582683859+0j)
22: (48.8043173715891+0j)
23: (48.836083639949756+0j)
24: (50.797444971643785+0j)
25: (51.14982961694209+0j)
26: (51.1571646618797+0j)
27: (51.929238779442336+0j)
28: (56.27354665831974+0j)
29: (56.29006328177593+0j)
30: (61.01783720542869+0j)


In [31]:
print(evalues)

(9.38769033434536+0j)
